# Ridge Tier-2b: finite-pool features with A3 gap-only base_rates

**Intent:** same finite-pool features as tier 2, but base_rates come from an A3 gap-only-weighted pool instead of A1 recency. Tests Jake's hypothesis that embargo-anchor similarity (gap matching) is the conceptually right pool definition for 'which critics review this KIND of movie'.

**A3 gap-only selector:** `combined_score_with_scores(..., α=1.0, σ_gap=8)`. With α=1.0 the Jaccard term is zero-weighted, so the selector reduces to `exp(−|gap_diff|/8)` ranking. Top-20 by that score become training; their scores become per-movie weights.

**Base_rate formula:** for each critic c in the union of training reviewers,
  `base_rate[c] = (Σ slug_weight[s] for s ∈ movies_reviewed_by_c) / n_movies`
where weights are normalized to sum to `n_movies` (same convention as ship's E2).

**Critical LOO guarantee:** the target itself is never in the training pool. `combined_score_with_scores` already filters `gaps['slug'] != target`, and we assert that the training_slugs list excludes the target before computing any features. Target's REVIEWS never enter the base_rate computation.

**Top-tier:** top 30 critics by A3-weighted base_rate within the A3 pool (target-adaptive).

**Features (3 new, parallel to tier 2):**
- `remaining_base_rate_sum_a3`
- `pool_mass_consumed_a3`
- `observed_top_tier_frac_a3`

Compared against: library, ship, ridge_orig, ridge_t1, ridge_t2 (A1), ridge_t2b (A3).

**Plan doc:** `brainstorm/brainstorm_ridge_optimization.md` § Tier 2 design caveats (base_rate source).


In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
if NB_DIR.name != 'notebooks':
    NB_DIR = NB_DIR / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import pickle
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold

import _helpers as H

print(f'cohort: {len(H.close_date_map)} movies')


## Noon-shift


In [ ]:
_day_mask = H.reviews['timestamp_confidence'] == 'd'
_n_shifted = int(_day_mask.sum())
H.reviews.loc[_day_mask, 'estimated_timestamp'] = (
    H.reviews.loc[_day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)
H.first_review_ts = (
    H.reviews[H.reviews['movie_slug'].isin(H.close_date_map)]
    .groupby('movie_slug')['estimated_timestamp'].min()
)
_new_first = H.first_review_ts.to_dict()
H.gaps['first_review_ts'] = H.gaps['slug'].map(_new_first)
H.gaps['gap_days'] = (
    H.gaps['close_ts'] - H.gaps['first_review_ts']
).dt.total_seconds() / 86400
H.gap_lookup = dict(zip(H.gaps['slug'], H.gaps['gap_days']))
print(f'noon-shift: {_n_shifted} day-level reviews')


## Load tier 2 cache (has tier 1 + tier 2 predictions)


In [ ]:
TIER2_CACHE = H.CACHE_DIR / 'phase1_ridge_tier2.pkl'
assert TIER2_CACHE.exists(), 'run phase1_ridge_tier2.ipynb first'
with open(TIER2_CACHE, 'rb') as f:
    df = pickle.load(f)
print(f'loaded {len(df)} rows  ·  columns: {len(df.columns)}')


## Configuration


In [ ]:
SNAP_DAYS_LIST = [5, 4, 3, 2, 1]
BASE_FEATURES = [
    'observed_count', 'first_review_dbc', 'target_gap', 'observed_rate',
    'rate_last_day', 'rate_first_day', 'top_critic_frac',
    'pub_diversity', 'pub_entropy', 'low_activity_frac',
]
TIER1_FEATURES = BASE_FEATURES + [
    'log_observed_count', 'log_rate_last_day', 'sqrt_rate_last_day', 'rate_delta',
]
TIER2B_FEATURES = ['remaining_base_rate_sum_a3', 'pool_mass_consumed_a3', 'observed_top_tier_frac_a3']
ALL_FEATURES = TIER1_FEATURES + TIER2B_FEATURES

A3_POOL_SIZE = 20
A3_SIGMA_GAP = 8.0
A3_ALPHA_SEL = 1.0  # pure gap-score weighting, no Jaccard
TOP_TIER_N = 30
ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
CV_FOLDS = 5
CV_SEED = 42
CACHE = H.CACHE_DIR / 'phase1_ridge_tier2b.pkl'


## Build A3 gap-only context per target

For each target: select top-20 training by `exp(-|gap_diff|/8)` (via combined_score with α=1). Compute weighted per-critic base_rate. **Assert target_slug is never in training_slugs.**


In [ ]:
def build_a3gap_context(target_slug):
    target_gap = H.gap_lookup.get(target_slug)
    if target_gap is None:
        return None

    # A3 α=1: pure gap-score; Jaccard term weighted zero, so target_critics/target_window_days
    # don't affect the ranking. Pass empty set + arbitrary 1.0 to make this explicit.
    scores = H.combined_score_with_scores(
        target=target_slug,
        target_gap=target_gap,
        target_critics=set(),
        target_window_days=1.0,
        k=A3_POOL_SIZE,
        alpha=A3_ALPHA_SEL,
        sigma_gap=A3_SIGMA_GAP,
    )
    if len(scores) < 5:
        return None

    training_slugs = list(scores.keys())
    # LOO guarantee: target must not be in training
    assert target_slug not in training_slugs, f'leakage: {target_slug} in A3 pool'

    n_movies = len(training_slugs)
    raw_weights = np.array([scores[s] for s in training_slugs], dtype=float)
    total_w = raw_weights.sum()
    if total_w <= 0:
        norm_weights = np.ones_like(raw_weights)
    else:
        norm_weights = raw_weights * (n_movies / total_w)
    slug_weight = dict(zip(training_slugs, norm_weights))

    # Build per-critic weighted base_rate.
    train = H.reviews[H.reviews['movie_slug'].isin(training_slugs)]
    # Paranoid check: target's reviews must not be in train
    assert (train['movie_slug'] == target_slug).sum() == 0, f'leakage: {target_slug} reviews in A3 train'

    base_rate = {}
    for name, group in train.groupby('reviewer_name'):
        movies_seen = group['movie_slug'].unique()
        weighted_ratio = float(sum(slug_weight[s] for s in movies_seen) / n_movies)
        base_rate[name] = weighted_ratio

    total_sum = float(sum(base_rate.values()))

    # Top tier by weighted base_rate
    sorted_critics = sorted(base_rate.items(), key=lambda p: -p[1])
    top_tier = set(c for c, _ in sorted_critics[:TOP_TIER_N])

    return {
        'training_slugs': training_slugs,
        'base_rate': base_rate,
        'total_sum': total_sum,
        'top_tier': top_tier,
        'weights': slug_weight,
    }


a3_cache = {}
start = time.time()
for slug in sorted(H.close_date_map):
    ctx = build_a3gap_context(slug)
    if ctx is not None:
        a3_cache[slug] = ctx
print(f'built A3 pool for {len(a3_cache)} targets  ·  {time.time()-start:.1f}s')

# Sanity: diagnostic
sample_slug = list(a3_cache.keys())[0]
sample = a3_cache[sample_slug]
print(f'\nexample: {sample_slug}  target_gap={H.gap_lookup[sample_slug]:.2f}d')
print(f'  training slugs (first 5): {sample["training_slugs"][:5]}')
print(f'  weights summary: min={min(sample["weights"].values()):.3f}  max={max(sample["weights"].values()):.3f}')
print(f'  critics in pool: {len(sample["base_rate"])}')
print(f'  total_sum: {sample["total_sum"]:.2f}')
print(f'  top_tier size: {len(sample["top_tier"])}')
sums = [c['total_sum'] for c in a3_cache.values()]
print(f'\ntotal_sum across targets: median={np.median(sums):.2f}  IQR={np.quantile(sums, 0.75) - np.quantile(sums, 0.25):.2f}')


## Diagnostic: how different are A1 and A3 pools?

If they overlap ≥80% for most targets, tier 2b will be essentially the same as tier 2.
If they diverge, A3 is bringing different critics into the pool.


In [ ]:
# Rebuild A1 pools to compare (same function as tier 2)
def build_a1_context(target_slug):
    close_ts = H.close_date_map[target_slug]
    training_slugs = H.default_training_slugs(
        H.movies, exclude_slug=target_slug, n=A3_POOL_SIZE, before_date=close_ts,
    )
    return training_slugs


overlaps = []
for slug in list(a3_cache.keys())[:50]:
    a3_set = set(a3_cache[slug]['training_slugs'])
    a1_set = set(build_a1_context(slug))
    if not a3_set or not a1_set:
        continue
    jacc = len(a3_set & a1_set) / len(a3_set | a1_set)
    overlaps.append(jacc)

print(f'A3-vs-A1 training-set Jaccard (n={len(overlaps)} sampled targets):')
print(f'  median={np.median(overlaps):.2f}  mean={np.mean(overlaps):.2f}  min={min(overlaps):.2f}  max={max(overlaps):.2f}')


## Compute finite-pool features using A3 base_rates


In [ ]:
def compute_pool_features_a3(target_slug, snap_days):
    ctx = a3_cache.get(target_slug)
    if ctx is None:
        return None
    close_ts = H.close_date_map[target_slug]
    midnight_utc = close_ts.floor('D')
    snap_time = midnight_utc - pd.Timedelta(days=snap_days)
    state = H.snapshot_state(target_slug, snap_time)
    if state is None:
        return None
    observed = state['observed_critics']

    base_rate = ctx['base_rate']
    total_sum = ctx['total_sum']
    top_tier = ctx['top_tier']

    obs_br_sum = float(sum(base_rate.get(c, 0.0) for c in observed))
    remaining_br_sum = max(total_sum - obs_br_sum, 0.0)
    pool_mass_consumed = (obs_br_sum / total_sum) if total_sum > 0 else 0.0
    top_observed = len(observed & top_tier)
    observed_top_tier_frac = top_observed / TOP_TIER_N

    return {
        'remaining_base_rate_sum_a3': remaining_br_sum,
        'pool_mass_consumed_a3': pool_mass_consumed,
        'observed_top_tier_frac_a3': observed_top_tier_frac,
    }


for feat in TIER2B_FEATURES:
    df[feat] = np.nan

start = time.time()
for idx, row in df.iterrows():
    feats = compute_pool_features_a3(row['target_slug'], int(row['snap_days']))
    if feats is None:
        continue
    for k, v in feats.items():
        df.at[idx, k] = v
print(f'computed A3 pool features in {time.time()-start:.1f}s')

print('\nA3 feature distributions at T-3d:')
print(df[df['snap_days'] == 3][TIER2B_FEATURES].describe().round(3).to_string())

print('\nA3 vs A1 feature correlation at T-3d (for reference):')
cmp = df[df['snap_days'] == 3][
    ['remaining_base_rate_sum', 'remaining_base_rate_sum_a3',
     'pool_mass_consumed', 'pool_mass_consumed_a3',
     'observed_top_tier_frac', 'observed_top_tier_frac_a3']
].corr()
print(cmp.round(3).to_string())


## α CV + LOO predict


In [ ]:
def select_alpha(X, y):
    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=CV_SEED)
    best_alpha, best_mae = None, np.inf
    for alpha in ALPHA_GRID:
        fold_errs = []
        for train_idx, test_idx in kf.split(X):
            pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
            pipe.fit(X[train_idx], y[train_idx])
            preds = pipe.predict(X[test_idx])
            fold_errs.extend(np.abs(preds - y[test_idx]).tolist())
        mae = float(np.mean(fold_errs))
        if mae < best_mae:
            best_mae, best_alpha = mae, alpha
    return best_alpha


def loo_predict(X, y, alpha):
    preds = np.zeros(len(X))
    for i in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[i] = False
        pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
        pipe.fit(X[mask], y[mask])
        preds[i] = pipe.predict(X[i:i+1])[0]
    return preds


snap_alpha_t2b = {}
df['ridge_t2b_pred'] = np.nan
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days].dropna(subset=ALL_FEATURES + ['actual'])
    if len(sub) < CV_FOLDS * 2:
        continue
    X = sub[ALL_FEATURES].values
    y = sub['actual'].values.astype(float)
    best_alpha = select_alpha(X, y)
    snap_alpha_t2b[snap_days] = best_alpha
    preds = loo_predict(X, y, best_alpha)
    df.loc[sub.index, 'ridge_t2b_pred'] = preds
    print(f'  T-{snap_days}d (n={len(sub)}): α*={best_alpha}')

with open(CACHE, 'wb') as f:
    pickle.dump(df, f)
print(f'saved {CACHE}')


## Per-variant cohort summary (6-way)


In [ ]:
def metrics(sub, pred_col):
    s = sub.dropna(subset=[pred_col])
    if len(s) == 0:
        return None
    err = s[pred_col].values - s['actual'].values
    return {
        'n': len(s), 'MAE': float(np.abs(err).mean()),
        'me': float(err.mean()),
        'p90_abs_err': float(np.quantile(np.abs(err), 0.9)),
    }


VARIANTS = [
    ('library',    'lib_pred'),
    ('ship',       'ship_pred'),
    ('ridge_orig', 'ridge_pred'),
    ('ridge_t1',   'ridge_t1_pred'),
    ('ridge_t2',   'ridge_t2_pred'),
    ('ridge_t2b',  'ridge_t2b_pred'),
]

print('=== cohort summary ===\n')
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days]
    print(f'T-{snap_days}d')
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            print(f'  {name:12s}  n={m["n"]:3d}  MAE={m["MAE"]:6.2f}  '
                  f'me={m["me"]:+6.2f}  p90|e|={m["p90_abs_err"]:6.2f}')
    print()


## h/m subset


In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm_df = df[df['target_slug'].isin(HM)]

print('=== h/m subset ===\n')
for snap_days in SNAP_DAYS_LIST:
    sub = hm_df[hm_df['snap_days'] == snap_days]
    if sub.empty:
        continue
    print(f'T-{snap_days}d (n={len(sub)})')
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            print(f'  {name:12s}  MAE={m["MAE"]:6.2f}  me={m["me"]:+6.2f}')
    print()


## Paired bootstrap (tier 2b vs baselines)


In [ ]:
COL_MAP = {name: col for name, col in VARIANTS}
PAIRS = [('ridge_orig', 'ridge_t2b'), ('ridge_t1', 'ridge_t2b'), ('ridge_t2', 'ridge_t2b')]

print('=== paired bootstrap ΔMAE (A − B, +ve → B wins) ===\n')
print(f'{"snap":<6}{"A":<12}{"B":<12}{"Δ":>10}{"CI95_lo":>10}{"CI95_hi":>10}{"Δ %":>8}{"n":>6}  result')
for snap_days in SNAP_DAYS_LIST:
    snap_df = df[df['snap_days'] == snap_days]
    for a, b in PAIRS:
        a_col, b_col = COL_MAP[a], COL_MAP[b]
        paired = snap_df.dropna(subset=[a_col, b_col, 'actual'])
        if len(paired) < 5:
            continue
        a_abs = np.abs(paired[a_col].values - paired['actual'].values)
        b_abs = np.abs(paired[b_col].values - paired['actual'].values)
        deltas = a_abs - b_abs
        point, lo, hi = H.bootstrap_mae_delta(deltas, n_boot=1000)
        pct = 100 * point / a_abs.mean() if a_abs.mean() else float('nan')
        if lo > 0:
            result = f'{b:>12} wins'
        elif hi < 0:
            result = f'{a:>12} wins'
        else:
            result = '    ns'
        print(f'T-{snap_days}d  {a:<12}{b:<12}{point:>+10.3f}{lo:>+10.3f}{hi:>+10.3f}{pct:>+8.2f}{len(paired):>6}  {result}')
    print()


## Coefficients — is A3-gap signal any different from A1?


In [ ]:
print('=== tier-2b coefficients (standardized) ===\n')
for snap_days in SNAP_DAYS_LIST:
    alpha = snap_alpha_t2b.get(snap_days)
    if alpha is None:
        continue
    sub = df[df['snap_days'] == snap_days].dropna(subset=ALL_FEATURES + ['actual'])
    if len(sub) < 10:
        continue
    X = sub[ALL_FEATURES].values
    y = sub['actual'].values.astype(float)
    pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    pipe.fit(X, y)
    coefs = pipe.named_steps['ridge'].coef_
    intercept = pipe.named_steps['ridge'].intercept_
    pairs = sorted(zip(ALL_FEATURES, coefs), key=lambda p: -abs(p[1]))
    print(f'T-{snap_days}d  α={alpha}  intercept={intercept:.2f}')
    for feat, c in pairs:
        marker = '  ← pool(A3)' if feat in TIER2B_FEATURES else ''
        print(f'  {feat:30s}  {c:+7.2f}{marker}')
    print()


## Plot


In [ ]:
import matplotlib.pyplot as plt

summary_rows = []
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days]
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            summary_rows.append({'snap_days': snap_days, 'variant': name, **m})
summary = pd.DataFrame(summary_rows)

fig, ax = plt.subplots(1, 1, figsize=(11, 4.5))
colors = {'library': 'tab:gray', 'ship': 'tab:red',
          'ridge_orig': 'tab:blue', 'ridge_t1': 'tab:green',
          'ridge_t2': 'tab:purple', 'ridge_t2b': 'tab:orange'}
markers = {'library': 's', 'ship': 'o', 'ridge_orig': '^',
           'ridge_t1': 'D', 'ridge_t2': '*', 'ridge_t2b': 'P'}
for name, _ in VARIANTS:
    sub = summary[summary['variant'] == name].sort_values('snap_days', ascending=False)
    ax.plot(sub['snap_days'], sub['MAE'], '-',
            marker=markers[name], color=colors[name], label=name, markersize=9)
ax.invert_xaxis()
ax.set_xlabel('snap days before close')
ax.set_ylabel('MAE (reviews)')
ax.set_title('Phase-1 MAE — tier-2b (A3 gap-only pool) vs all other variants')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Observations

*(fill in after run)*
